In [1]:
import numpy as np
import torchvision as thv

from torch import torch
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler, SequentialSampler

from utils import Utils
from consts import Consts
from feed_fwd_nn_model import FeedFwdNNModel

In [ ]:
data_train = thv.datasets.MNIST(
    root="./data"
    , train=True
    , download=True
    , transform=thv.transforms.ToTensor()
)

I_train = np.random.randint(
    low=0
    , high=data_train.data.shape[0]
    , size=int(data_train.data.shape[0] * Consts.DS_SAMPLING_RATIO)
)

loader_train = DataLoader(
    dataset=data_train
    , batch_size=Consts.MINI_BATCH_SIZE
    , sampler=SubsetRandomSampler(I_train)
)

print(I_train.shape)
print(len(loader_train.sampler))

In [ ]:
data_val = thv.datasets.MNIST(
    root="./data"
    , train=False
    , download=True
    , transform=thv.transforms.ToTensor()
)

I_val = np.arange(0, int(data_val.data.shape[0] * Consts.DS_SAMPLING_RATIO))

loader_val = DataLoader(
    dataset=data_val
    , batch_size=Consts.MINI_BATCH_SIZE
    , sampler=SequentialSampler(I_val)
)

print(I_val.shape)
print(len(loader_val.sampler))

In [ ]:
lrs = [0.1, 0.01]
dropout_ps = [0.1, 0.2, 0.5]
optimizer_algs = ["sgd", "adam"]
hidden_sizes = [[], [500], [1000], [1000, 500]]

models = [
    FeedFwdNNModel(
        lr=lr
        , dropout_p=dropout_p
        , loader_val=loader_val
        , hidden_sizes=hidden_size
        , loader_train=loader_train
        , optimizer_alg=optimizer_alg
        , device_name=Utils.get_device_name()
    ) for lr in lrs for optimizer_alg in optimizer_algs for dropout_p in dropout_ps for hidden_size in hidden_sizes
]

In [ ]:
models_idps = [None] * len(models)

for model_idx, model in enumerate(models):
    models_idps[model_idx] = model.train_and_validate()

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=20
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , fig_size=(2 * 2 * 2 * 3, 2 * 2 * 3)
    , title=f"Training & Validation Losses"
    , x=[idp.iter_idx for idp in models_idps[0]]
    , yss_legend=[[f"{loss_type} of {model}" for loss_type in ["training", "validation"]] for model in models]
    , yss=[[[model_idp.training_loss for model_idp in model_idps], [model_idp.validation_loss for model_idp in model_idps]] for model_idps in models_idps]
)

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=20
    , y_axis_label="Error"
    , x_axis_label="Iterations"
    , fig_size=(2 * 2 * 2 * 3, 2 * 2 * 3)
    , title=f"Training & Validation Errors"
    , x=[idp.iter_idx for idp in models_idps[0]]
    , yss_legend=[[f"{loss_type} of {model}" for loss_type in ["training", "validation"]] for model in models]
    , yss=[[[model_idp.training_error for model_idp in model_idps], [model_idp.validation_error for model_idp in model_idps]] for model_idps in models_idps]
)